# 1. Data preprocessing

Neurotransformer classifies **single-channel EEG windows of 400 samples** (2 s at 200 Hz). Every dataset is
converted to one `.npz` file per window, stored as `data/<dataset>/{train,eval,test}/`:

| Dataset | Keys | Raw labels → classes |
|---|---|---|
| NMT events | `data` (1, 400), `ab_label` | 0 → Normal, {1, 3} → SW, 2 → SSW |
| Bonn | `data` (1, 400), `ab_label` | sets A/Z, B/O → Normal · C/N, D/F → Interictal · E/S → Ictal |
| TUEV | `x` (1, 400), `y` | 1 → Epileptic (SPSW, GPED, PLED overlap > τ) · 0 → Non-epileptic (EYEM, ARTF, BCKG) |

Set the paths in the next cell to wherever you downloaded the raw data. Sections for missing datasets are skipped.

In [ ]:
from pathlib import Path

# Run from the repository root or from notebooks/.
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = REPO / "data"

BONN_RAW_DIR = Path("/path/to/bonn")         # folders/files for the Z, O, N, F, S sets (*.txt)
TUEV_ROOT = Path("/path/to/tuev/v2.0.1")     # contains edf/train and edf/eval

## Bonn

Each of the 500 Bonn recordings (4097 samples at 173.61 Hz) is resampled to 200 Hz, band-pass filtered between
1 and 45 Hz (Hamming-windowed FIR) and cut into 2 s windows with 50 % overlap. Recordings are assigned to splits **before** windowing, so no recording contributes windows to two
splits. `configs/splits/bonn_split.json` holds the exact record split used in the experiments (350 / 75 / 75 records,
giving 7700 / 1650 / 1650 windows).

In [ ]:
from neurotransformer.data.preprocessing import preprocess_bonn

if BONN_RAW_DIR.exists():
    preprocess_bonn(str(BONN_RAW_DIR), str(DATA_DIR / "bonn"),
                    split_file=str(REPO / "configs" / "splits" / "bonn_split.json"))
else:
    print(f"Skipping Bonn: {BONN_RAW_DIR} not found")

## TUEV

Every EDF in the official `train` and `eval` sets is clipped to ±100 µV, resampled to 200 Hz, band-pass filtered
(1–45 Hz FIR), cleaned with ICA (muscle and eye components) and converted to the 22-channel TCP montage used by
the `.rec` annotations. Each channel is cut into 2 s windows with 50 % overlap. A window is epileptic when it
overlaps an SPSW/GPED/PLED annotation on that channel by more than τ = 0.25 of its length, non-epileptic when it
overlaps other annotations only, and dropped otherwise. ICA makes this slow; set `ica=False` for a quick test.
Requires `mne`.

In [ ]:
from neurotransformer.data.preprocessing import preprocess_tuev

if TUEV_ROOT.exists():
    preprocess_tuev(str(TUEV_ROOT), str(DATA_DIR / "tuev"), tau=0.25, ica=True)
else:
    print(f"Skipping TUEV: {TUEV_ROOT} not found")

## NMT events

The NMT-events windows are derived from the NMT Scalp EEG dataset
using event annotations. The training code expects `data/nmt/{train,eval,test}/<recording>_<...>.npz` with keys
`data` (1, 400) and `ab_label`. The recording-level plots in `04_visualization.ipynb` also need `channel`,
`beg` and `end` (sample indices at 200 Hz).

## Inspect the processed data

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from neurotransformer.data import NpzEEGDataset, available_splits

DATASET = "bonn"  # "nmt", "bonn" or "tuev"
root = DATA_DIR / DATASET

for split in available_splits(root):
    ds = NpzEEGDataset(str(root / split), DATASET)
    counts = np.bincount(ds.targets(), minlength=ds.spec.n_classes)
    print(f"{split:5s} {len(ds):7d} windows  " +
          "  ".join(f"{n}={c}" for n, c in zip(ds.spec.class_names, counts)))

In [ ]:
ds = NpzEEGDataset(str(root / "train"), DATASET)
targets = ds.targets()
fig, axes = plt.subplots(ds.spec.n_classes, 1, figsize=(10, 2.2 * ds.spec.n_classes), sharex=True)
for c, ax in enumerate(axes):
    x, _ = ds[int(np.flatnonzero(targets == c)[0])]
    ax.plot(x[0].numpy(), linewidth=0.8)
    ax.set_ylabel(ds.spec.class_names[c])
axes[-1].set_xlabel("Sample")
fig.suptitle(f"{DATASET}: one training window per class")
plt.tight_layout()